In [ ]:
from pathlib import Path
import sys

src_dir = next(p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "common.py").is_file() and (p / "feature_preference").is_dir())
sys.path.insert(0, str(src_dir))


In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision.transforms.v2 as transforms
from torch.utils.data import Dataset
import random
import hydra
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import DataLoader
from setproctitle import setproctitle
from tqdm import tqdm
import os
import random

import torchvision
from common import *  # from your_loss_file import BTLoss

# --- Synthetic Dataset ---
class SyntheticColorDataset(Dataset):
    def __init__(self, size_dataset, crop_min, split='train', transform=None, no_aug=False):
        self.size_dataset = size_dataset
        self.split = split
        self.no_aug = no_aug

        # Dimensions
        self.m = 96
        self.a = 16
        self.b = 16

        self.colors_center = torch.rand(5, 3)
        self.colors_outer = torch.rand(5, 3)

        # Pre-generate indices: (background_idx, center_idx, outer_idx)
        self.data_info = []
        self.datas = []
        mean = 0.0
        var = 0.0
        for _ in range(size_dataset):
            center_idx = random.randint(0, len(self.colors_center)-1)
            outer_idx = random.randint(0, len(self.colors_outer)-1)
            self.data_info.append((center_idx, outer_idx))
            imgs = self._generate_image(center_idx, outer_idx)
            self.datas.append(imgs)
            image_preprocess = transforms.ToTensor()(imgs)
            mean += image_preprocess.mean(dim=[1, 2])
            var += image_preprocess.var(dim=[1, 2])

        mean /= len(self.datas)
        var /= len(self.datas)
        std = torch.sqrt(var)
        print(mean, std)

        # Transforms
        # mean, std = [0.5, 0.5, 0.5], [0.5, 0.5, 0.5]
        self.transform = transform
        if self.transform is None:
            self.transform = transforms.Compose([
                transforms.ToTensor(),
                transforms.Normalize(mean, std),
            ])

        min_scale = crop_min
        print(min_scale)
        self.augment = transforms.Compose([
            transforms.ToTensor(),
            transforms.RandomApply([transforms.GaussianNoise(sigma=0.05)], p=0.5),
            transforms.ToPILImage(),
            transforms.RandomResizedCrop(self.m, scale=(min_scale, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomApply([transforms.ColorJitter(0.4, 0.4, 0.2, 0.1)], p=0.8),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
        self.augment_prime = transforms.Compose([
            transforms.ToTensor(),
            transforms.RandomApply([transforms.GaussianNoise(sigma=0.05)], p=0.5),
            transforms.ToPILImage(),
            transforms.RandomResizedCrop(self.m, scale=(min_scale, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomApply([transforms.ColorJitter(0.4, 0.4, 0.2, 0.1)], p=0.8),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])

    def _generate_image(self, center_idx, outer_idx):
        # Background
        img = torch.zeros(3, self.m, self.m)

        radius = round(math.sqrt(self.a * self.a / math.pi))

        y = torch.arange(self.m)
        x = torch.arange(self.m)
        yy, xx = torch.meshgrid(y, x, indexing='ij')

        jitter_range = 8
        offset_y = random.randint(-jitter_range, jitter_range)
        offset_x = random.randint(-jitter_range, jitter_range)

        cy = (self.m // 2) + offset_y
        cx = (self.m // 2) + offset_x

        dist_sq = (xx - cx) ** 2 + (yy - cy) ** 2
        mask = dist_sq <= radius ** 2

        for c in range(3):
            img[c, mask] = self.colors_center[center_idx][c]

        # # Center Square (ax8)
        # start_a = (self.m - self.a) // 2
        # img[:, start_a:start_a + self.a, start_a:start_a + self.a] = self.colors_center[center_idx].view(3, 1, 1)

        # Outer Square (bxb) - positioned at top-left with slight margin
        margin_x = 10
        margin_y = 10
        corners = [
            (margin_x, margin_y),  # Top-Left
            (margin_x, self.m - self.b - margin_y),  # Bottom-Left
            (self.m - self.b - margin_x, margin_y),  # Top-Right
            (self.m - self.b - margin_x, self.m - self.b - margin_y)  # Bottom-Right
        ]
        start_y, start_x = random.choice(corners)
        img[:, start_y:start_y + self.b, start_x:start_x + self.b] = self.colors_outer[outer_idx].view(3, 1, 1)

        return transforms.ToPILImage()(img)

    def __len__(self):
        return len(self.data_info)

    def __getitem__(self, idx):
        center_idx, outer_idx = self.data_info[idx]
        image_pil = self._generate_image(center_idx, outer_idx)

        # Label1: center color (main task), Label2: outer color (shortcut)

        if self.no_aug:
            return self.transform(image_pil), center_idx, outer_idx
        if self.split == 'train':
            return self.augment(image_pil), self.augment_prime(image_pil)
        else:
            return self.transform(image_pil), center_idx, outer_idx

def get_synthetic_dataloaders(crop_min, batch_size=64):
    bs = batch_size

    # Define sizes
    train_len = 100
    test_len = 20

    train_ds = SyntheticColorDataset(train_len, crop_min, split='train', no_aug=False)
    test_ds = SyntheticColorDataset(test_len, crop_min, split='test', no_aug=True)
    train_noaug_ds = SyntheticColorDataset(train_len, crop_min, split='train', no_aug=True)

    train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True, num_workers=4, pin_memory=True, drop_last=True)
    test_loader = DataLoader(test_ds, batch_size=512, shuffle=False, num_workers=4, pin_memory=True)
    train_noaug_loader = DataLoader(train_noaug_ds, batch_size=bs, shuffle=True, num_workers=4, pin_memory=True)

    return train_loader, test_loader, train_noaug_loader

In [ ]:
import matplotlib.pyplot as plt

train_loader, test_loader, train_noaug_loader = get_synthetic_dataloaders(crop_min = 1.0)
# tensor to pil

fig, axs = plt.subplots(1, 5, figsize=(24, 6.5))

for i in range(5):
    _, (data1, data2) = next(enumerate(train_loader))
    img_tensor = data1[i,:,:,:].cpu().detach()
    img_numpy = img_tensor.permute(1, 2, 0).numpy()
    axs[i].imshow((img_numpy + 1 / 2))
    axs[i].axis('off')
plt.tight_layout()
plt.savefig('figure/rrc_toy_dataset.pdf')
plt.show()

In [ ]:
import os

def load_from_csv(path):
    output = []
    if not os.path.exists(path):
        return np.array(output)
    with open(path, 'r') as f:
        for line in f:
            output.append(float(line.strip()))
    return np.array(output)


def load_from_csv_split(path):
    output = []
    if not os.path.exists(path):
        return np.array(output)
    with open(path, 'r') as f:
        for line in f:
            split = line.strip().split(',')
            if len(output) == 0:
                for i in range(len(split)):
                    output.append([])
            for i in range(len(split)):
                output[i].append(float(split[i].strip()))
    for i in range(len(output)):
        output[i] = np.array(output[i])
    return output

In [ ]:
lambdas = 0.05
method = "rrc_color_agg"
directory = f"datas/{method}_{lambdas}/2"
loss_agg = load_from_csv(f"{directory}/loss.csv")
[acc_center_agg, acc_outer_agg, step_agg, loss_center_agg, loss_outer_agg] = load_from_csv_split(f"{directory}/accuracy.csv")
eigvals_agg = load_from_csv_split(f"{directory}/eig.csv")


# method = "rrc_color_middle"
# directory = f"datas/{method}_{lambdas}/0"
# loss_middle = load_from_csv(f"{directory}/loss.csv")
# [acc_center_middle, acc_outer_middle, step_middle, loss_center_middle, loss_outer_middle] = load_from_csv_split(f"{directory}/accuracy.csv")
# eigvals_middle = load_from_csv_split(f"{directory}/eig.csv")

method = "rrc_color_rough"
directory = f"datas/{method}_{lambdas}/2"
loss_rough = load_from_csv(f"{directory}/loss.csv")
[acc_center_rough, acc_outer_rough, step_rough, loss_center_rough, loss_outer_rough] = load_from_csv_split(f"{directory}/accuracy.csv")
eigvals_rough = load_from_csv_split(f"{directory}/eig.csv")

In [ ]:
plt.plot(step_agg, acc_center_agg, alpha=0.5, linewidth=3, color='red', label='RRC ratio : [0.08, 1.0], Center Circle')
plt.plot(step_rough, acc_center_rough, alpha=0.5, linewidth=3, color='green', label='RRC ratio : [0.7, 1.0], Center Circle')

plt.ylim(10-0.4, 100.4)
plt.xlabel('Steps')
plt.ylabel('Test Error.')
# plt.xlim(0, 40)
# plt.yscale('log')
plt.legend(loc='lower right')
plt.show()

plt.plot(step_agg, acc_outer_agg, alpha=0.5, linewidth=2, color='red', label='RRC ratio : [0.08, 1.0], Outer Square') #RRC ratio : [0.08, 1.0]
plt.plot(step_rough, acc_outer_rough, alpha=0.5, linewidth=2, color='green', label='RRC ratio : [0.7, 1.0], Outer Square')

plt.ylim(10-0.4, 100.4)
plt.xlabel('Steps')
plt.ylabel('Test Error.')
# plt.xlim(0, 40)
# plt.yscale('log')
plt.legend(loc='lower right')
plt.show()

In [ ]:
import seaborn as sns

device = torch.device("cuda")
sns.set_theme(style="whitegrid", font="cmr10", font_scale=1.9)
plt.rcParams["axes.formatter.use_mathtext"] = True
plt.rcParams.update({"pdf.fonttype": 42, "ps.fonttype": 42})
colors = sns.color_palette()
colors

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

lambdas = 0.05
seeds = [0, 1, 2, 3, 4, 5, 6]

def process_seeds(method, lambdas, seeds):
    acc_centers = []
    acc_outers = []
    steps_list = []

    for seed in seeds:
        directory = f"datas/{method}_{lambdas}/{seed}"
        data = load_from_csv_split(f"{directory}/accuracy.csv")
        acc_centers.append(data[0])
        acc_outers.append(data[1])
        steps_list.append(data[2])

    min_len = min(len(s) for s in steps_list)

    out_step = steps_list[0][:min_len]

    # Crop and stack
    cropped_centers = 100-np.array([x[:min_len] for x in acc_centers])
    cropped_outers = 100-np.array([x[:min_len] for x in acc_outers])

    # Calculate Mean and Std
    out_acc_center = np.mean(cropped_centers, axis=0)
    out_std_center = np.std(cropped_centers, axis=0)

    out_acc_outer = np.mean(cropped_outers, axis=0)
    out_std_outer = np.std(cropped_outers, axis=0)

    return out_acc_center, out_std_center, out_acc_outer, out_std_outer, out_step

acc_center_agg, std_center_agg, acc_outer_agg, std_outer_agg, step_agg = process_seeds("rrc_color_agg", lambdas, seeds)
acc_center_rough, std_center_rough, acc_outer_rough, std_outer_rough, step_rough = process_seeds("rrc_color_rough", lambdas, seeds)

fig, axs = plt.subplots(1, 2, figsize=(24, 7), sharex=True, sharey=True)

# Plot 1
axs[0].plot(step_agg, acc_center_agg, alpha=0.5, linewidth=3, color='red', label='RRC ratio : [0.08, 1.0], Center Circle')
axs[0].fill_between(step_agg, acc_center_agg - std_center_agg, acc_center_agg + std_center_agg, color='red', alpha=0.2)

axs[0].plot(step_rough, acc_center_rough, alpha=0.5, linewidth=3, color='green', label='RRC ratio : [0.7, 1.0], Center Circle')
axs[0].fill_between(step_rough, acc_center_rough - std_center_rough, acc_center_rough + std_center_rough, color='green', alpha=0.2)

# plt.ylim(10-0.4, 100.4)
axs[0].set_xlabel('Steps')
axs[0].set_ylabel('Test Error')
# axs[0].set_yscale('log')
axs[0].legend(loc='upper right')

# Plot 2
axs[1].plot(step_agg, acc_outer_agg, alpha=0.5, linewidth=2, color='red', label='RRC ratio : [0.08, 1.0], Outer Square')
axs[1].fill_between(step_agg, acc_outer_agg - std_outer_agg, acc_outer_agg + std_outer_agg, color='red', alpha=0.2)

axs[1].plot(step_rough, acc_outer_rough, alpha=0.5, linewidth=2, color='green', label='RRC ratio : [0.7, 1.0], Outer Square')
axs[1].fill_between(step_rough, acc_outer_rough - std_outer_rough, acc_outer_rough + std_outer_rough, color='green', alpha=0.2)

# plt.ylim(10-0.4, 100.4)
axs[1].set_xlabel('Steps')
# axs[1].set_ylabel('Test Error')
axs[1].legend(loc='upper right')
# plt.yscale('log')
plt.tight_layout()
plt.savefig('figure/rrc_toy.pdf')
plt.show()

In [ ]:
for i in range(len(eigvals_agg)):
    plt.plot(eigvals_agg[i])
plt.show()

In [ ]:
for i in range(len(eigvals_middle)):
    plt.plot(eigvals_middle[i])
plt.show()

In [ ]:
for i in range(len(eigvals_rough)):
    plt.plot(eigvals_rough[i])
plt.show()